# Chapter 9: Measuring Agent Observability, Cost, and ROI

This notebook mirrors the chapter. Run the cells in order, from the `Chapter09`
folder. Nothing here needs an API key. The SDK runner, the tools, the meter, the
run budget, the guardrail, and the ledger are real. Only the model's replies and
the token counts it reports are scripted, which makes every dollar figure the same
on every run.

All plant data is synthetic. Prices come from `data/price-card.yaml`.

Pinned version: openai-agents 0.24.0.

In [1]:
import sys
from importlib.metadata import version
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
print("openai-agents", version("openai-agents"))

openai-agents 0.24.0


## 1. What one approved work order costs

Table 9.1, computed from the month's ledger.

In [2]:
!{sys.executable} -m examples.unit_cost

the quarter of the budget review, as a monthly average: 6,000 attempts

  model tokens                $  0.34
  tools and retrieval         $  0.06
  checks and sampled judge    $  0.05
  trace storage               $  0.01
  human review                $  2.94
  direct cost per attempt     $  3.40
  approved by a planner           71%   (4,260 of 6,000)
  cost per approved order     $  4.79
  human baseline, same work   $  4.20
  agent against baseline         +14%

by severity          attempts  approved   per order  baseline
  complex               1,000      35%      16.18      7.60  above baseline
  moderate              2,000      68%       5.81      4.60  above baseline
  routine               3,000      85%       2.68      2.80  below baseline


## 2. The price card and the meter

Every cost comes from one versioned file. The meter splits a usage report into five
lines. Reasoning tokens are part of the output count, so they are priced once and
shown on their own line.

In [3]:
from dataclasses import asdict

from chapter9.meter import price_usage, usage_of
from chapter9.pricecard import UnknownPrice, load_price_card

card = load_price_card()
print("price card", card.version, card.rates("gpt-6.2-sol"))
cost = price_usage(
    usage_of(10_000, 2_000, cached=6_000, written=1_000, reasoning=1_200),
    card.rates("gpt-6.2-sol"),
)
for line, usd in asdict(cost).items():
    print(f"  {line:<15}${usd:.4f}")
print(f"  {'total':<15}${cost.total:.4f}")
try:
    card.rates("gpt-7-preview")
except UnknownPrice as error:
    print(error)

price card 2026-11-24 Rates(input=2.0, cached_input=0.1, cache_write=2.5, output=10.0)
  uncached_input $0.0060
  cached_input   $0.0006
  cache_write    $0.0025
  reasoning      $0.0120
  visible_output $0.0080
  total          $0.0291
model 'gpt-7-preview' is not on price card 2026-11-24


## 3. One run, span by span

The same work twice: every step on the frontier model with nothing cached, then
with a cached prefix and a model chosen for each step. Look for the repeated call.

In [4]:
!{sys.executable} -m examples.one_run

work order for press-07: main drive bearing wear
plant plant-2, severity routine, for planner:okafor
agent 40.6, prompts maintenance@40

every step on the frontier model, nothing cached
  step   model or tool          input  cached  output  reasoning     cost
  plan   gpt-6.2-sol            8,000       0   2,400      1,800  $0.0400
  tool   read_repair_history                                      $0.0100
  tool   read_sensor_summary                                      $0.0100
  read   gpt-6.2-sol           14,000       0     600        300  $0.0340
  tool   check_parts                                              $0.0200
  read   gpt-6.2-sol           15,000       0     500        250  $0.0350
  tool   check_parts          repeat of the call above            $0.0200
  read   gpt-6.2-sol           16,000       0     500          0  $0.0370
  tool   search_manual                                            $0.0200
  read   gpt-6.2-sol           19,000       0     800        500  $0.0460


## 4. A call across the border

Your own record of a call to a supplier's agent.

In [5]:
!{sys.executable} -m examples.border

reply: completed, sent with {'x-correlation-id': 'WO-2R0412'}
  span             border.nordbolt
  case_id          WO-2R0412
  counterparty     nordbolt
  remote_task_id   task-8841
  reported_version 2.3.1
  status           completed
  attempts         2
  latency_ms       under a second
  quoted_cost_usd  0.04
  receipt_id       nb-rcpt-55120


## 5. The anatomy of one turn

Figure 9.3 is drawn from this table.

In [6]:
!{sys.executable} -m examples.one_turn

turn 8 on gpt-6.1-terra, price card 2026-11-24
reads 35,000 tokens: instructions 1,500, tool definitions 12,000,
  history 20,000, new material 1,500
writes 2,000 tokens, of which 1,200 are reasoning

                                   input  cached   write  reason  answer   total
nothing cached                    0.0350  0.0000  0.0000  0.0060  0.0040  0.0450
tools deferred, nothing cached    0.0250  0.0000  0.0000  0.0060  0.0040  0.0350
prefix cached                     0.0015  0.0016  0.0025  0.0060  0.0040  0.0156
prefix changed, cache rewritten   0.0015  0.0000  0.0419  0.0060  0.0040  0.0534


## 6. Why the bill rose, and whether the ledger matches the invoice

Runs, tokens per run, and price per token, then the reconciliation, then the
production metrics for the two months.

In [7]:
!{sys.executable} -m examples.why_bill_rose

quarter on quarter, monthly averages
  runs               x2.00
  tokens per run     x3.00
  price per token    x0.50
  token bill         x3.00   $680 to $2,040

ledger $2,040.00 against invoice $2,071.30: gap 1.5%, within the 3% tolerance



the quarter of the budget review, as a monthly average
  turns per run, median and 95th percentile  8 and 18
  runs with a repeated call that executed    9.2%
  runs with a repeated call that was refused 0.0%
  tool calls that failed                     0.5%
  input tokens read from the cache           0%
  tokens per run                             130,000



one quarter later, as a monthly average
  turns per run, median and 95th percentile  5 and 10
  runs with a repeated call that executed    0.0%
  runs with a repeated call that was refused 5.1%
  tool calls that failed                     0.5%
  input tokens read from the cache           83%
  tokens per run                             101,333


## 7. The run budget

A model that keeps asking the same question. The guardrail refuses the repeats, and
the budget stops the run before the call that would exceed it.

In [8]:
!{sys.executable} -m examples.budget_stop

run budget $0.20, 8 model calls, 8 tool calls


  step   model or tool          input  cached  output  reasoning     cost
  plan   gpt-6.2-sol            8,000       0   2,400      1,800  $0.0400
  tool   read_repair_history                                      $0.0100
  tool   read_sensor_summary                                      $0.0100
  read   gpt-6.2-sol           14,000       0     600        300  $0.0340
  tool   check_parts                                              $0.0200
  read   gpt-6.2-sol           15,000       0     500        250  $0.0350
  tool   check_parts          refused: repeat                     $0.0000
  read   gpt-6.2-sol           15,000       0     500        250  $0.0350
  tool   check_parts          refused: repeat                     $0.0000
  model $0.1440  tools $0.0400  turns 5  repeated calls 2
stopped: run budget exhausted: spent $0.1840 of $0.20, next action needs $0.0400
  outcome stopped, model calls left 4

a child asks for $0.50 from a parent that has $0.10 left
the child is given $0.10


Try it yourself: give the run a larger budget and a smaller number of model calls,
and see which limit stops it.

In [9]:
from chapter9.budget import BudgetExceeded, RunBudget
from chapter9.planner import RunContext, plan_work_order
from examples.common import frontier_agents, start
from examples.scripted import MACHINE, TAGS, stuck_run

card, ledger = start()
budget = RunBudget(limit_usd=5.00, model_calls=4, tool_calls=8)
agents = frontier_agents(*stuck_run(), card, budget, guarded=True)
try:
    await plan_work_order(MACHINE, agents, RunContext(budget), TAGS)
except BudgetExceeded as stop:
    print("stopped:", stop)

stopped: no model calls left: spent $0.1840 of $5.00, next action needs $0.0400


## 8. Where the model spend goes

By step and by model, before and after.

In [10]:
!{sys.executable} -m examples.token_share

the quarter of the budget review, as a monthly average
  every step on the frontier model, nothing cached
  step   model            input tokens  cached  reasoning    spend  share
  read   gpt-6.2-sol            588 M      0%     16.3 M    1,440    71%
  plan   gpt-6.2-sol             64 M      0%     15.9 M      340    17%
  draft  gpt-6.2-sol             68 M      0%      3.1 M      260    13%



one quarter later, as a monthly average
  cached prefix, a model for each step, repeats refused, complex orders off
  step   model            input tokens  cached  reasoning    spend  share
  plan   gpt-6.2-sol             42 M     71%      9.9 M      159    52%
  read   gpt-6.1-terra          330 M     85%      3.9 M      143    47%
  draft  gpt-6-luna              48 M     78%      0.0 M        5     2%



## 9. Cheaper is not the same as better value

Three configurations compared on cost per approved order. Figure 9.4 is drawn from
this table.

In [11]:
!{sys.executable} -m examples.frontier

configuration                model  attempt  approved  per order  baseline


before, 6,000 attempts        0.34     3.40       71%       4.79      4.20
  every step on the frontier model, nothing cached


candidate, 6,000 attempts     0.22     3.28       65%       5.05      4.20
  reading tool results moved to the mid-tier model


after, 4,500 attempts         0.07     2.64       83%       3.17      3.40
  cached prefix, a model for each step, repeats refused, complex orders off


## 10. The finance view

Cost per approved work order by plant, as it first looks and then like for like,
with an interval on every rate and a verdict that uses both ends of it.

In [12]:
!{sys.executable} -m examples.by_plant

the quarter of the budget review, as a monthly average
  by plant           tries  rate  95% interval   cost    base
  plant-1              1,600   62%  60% to 65%    6.19    4.86  above baseline
  plant-2              1,560   75%  73% to 78%    4.19    3.87  above baseline
  plant-3              1,500   73%  71% to 76%    4.49    4.04  above baseline
  plant-4              1,340   74%  71% to 76%    4.43    3.97  above baseline
  by plant and severity
  plant-1    complex     500   34%  30% to 38%   16.66    7.60  above baseline
  plant-1    moderate    500   66%  62% to 70%    5.99    4.60  above baseline
  plant-1    routine     600   83%  80% to 86%    2.75    2.80  at baseline
  plant-2    complex     160   36%  29% to 44%   15.62    7.60  above baseline
  plant-2    moderate    500   69%  65% to 73%    5.73    4.60  above baseline
  plant-2    routine     900   86%  84% to 88%    2.65    2.80  below baseline
  plant-3    complex     200   36%  30% to 43%   15.73    7.60  above ba

one quarter later, as a monthly average
  by plant           tries  rate  95% interval   cost    base
  plant-1                975   81%  79% to 84%    3.34    3.49  below baseline
  plant-2              1,275   84%  82% to 86%    3.05    3.33  below baseline
  plant-3              1,175   84%  82% to 86%    3.12    3.37  below baseline
  plant-4              1,075   83%  81% to 85%    3.20    3.43  below baseline
  by plant and severity
  plant-1    moderate    375   79%  75% to 83%    4.62    4.60  at baseline
  plant-1    routine     600   83%  80% to 86%    2.57    2.80  below baseline
  plant-2    moderate    375   81%  77% to 84%    4.52    4.60  at baseline
  plant-2    routine     900   86%  84% to 88%    2.48    2.80  below baseline
  plant-3    moderate    375   80%  76% to 84%    4.55    4.60  at baseline
  plant-3    routine     800   86%  83% to 88%    2.49    2.80  below baseline
  plant-4    moderate    375   80%  76% to 84%    4.56    4.60  at baseline
  plant-4    rout

## 11. Live models

`python -m examples.live_check` runs the planner against live models and checks
that the provider's usage report has the fields the meter prices. It needs
`OPENAI_API_KEY` and costs a few cents, so it is not run here.